In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical

np.random.seed(42)

In [ ]:
# CREATE SYNTHETIC DATASET
n = 5000

data = pd.DataFrame({
    'tenure': np.random.randint(1, 72, n),
    'monthly_charges': np.random.uniform(20, 120, n),
    'contract': np.random.choice(['month-to-month', 'one-year', 'two-year'], n),
    'internet_service': np.random.choice(['DSL', 'Fiber', 'None'], n),
    'support_calls': np.random.randint(0, 10, n),
    'payment_method': np.random.choice(['credit_card', 'bank_transfer', 'paypal'], n),
})

# Create churn probability
data['churn'] = (
    (data['tenure'] < 12).astype(int) +
    (data['monthly_charges'] > 80).astype(int) +
    (data['support_calls'] > 5).astype(int)
)

data['churn'] = (data['churn'] > 1).astype(int)

print(data.head())

In [ ]:
# EDA
print(data.info())
print(data.describe())

sns.countplot(x='churn', data=data)
plt.title("Churn Distribution")
plt.show()

sns.boxplot(x='churn', y='monthly_charges', data=data)
plt.show()

sns.boxplot(x='churn', y='tenure', data=data)
plt.show()

In [ ]:
# PREPROCESSING
le = LabelEncoder()

for col in ['contract', 'internet_service', 'payment_method']:
    data[col] = le.fit_transform(data[col])

X = data.drop('churn', axis=1)
y = data['churn']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)


In [ ]:
# BASELINE MODEL (LOGISTIC REGRESSION)
log_model = LogisticRegression()
log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)

print("\nLogistic Regression Accuracy:", accuracy_score(y_test, y_pred_log))
print(classification_report(y_test, y_pred_log))


In [ ]:
# TREE-BASED MODELS
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

rf = RandomForestClassifier(n_estimators=100)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("\nDecision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))


In [ ]:
# MODEL COMPARISON
results = {
    'Logistic': accuracy_score(y_test, y_pred_log),
    'Decision Tree': accuracy_score(y_test, y_pred_dt),
    'Random Forest': accuracy_score(y_test, y_pred_rf)
}
sns.barplot(x=list(results.keys()), y=list(results.values()))
plt.title("Model Comparison")
plt.ylabel("Accuracy")
plt.show()

In [ ]:
# HYPERPARAMETER TUNING (RANDOM FOREST)
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10]
}

grid = GridSearchCV(RandomForestClassifier(), param_grid, cv=3)
grid.fit(X_train, y_train)
best_rf = grid.best_estimator_

y_pred_best = best_rf.predict(X_test)

print("\nBest RF Accuracy:", accuracy_score(y_test, y_pred_best))
print("Best Params:", grid.best_params_)

In [ ]:
# CONFUSION MATRIX
cm = confusion_matrix(y_test, y_pred_best)

sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# DEEP LEARNING MODEL (ANN)
y_train_cat = to_categorical(y_train)
y_test_cat = to_categorical(y_test)

model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(2, activation='softmax')
])
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
history = model.fit(
    X_train, y_train_cat,
    validation_data=(X_test, y_test_cat),
    epochs=30,
    batch_size=32,
    verbose=1
)

In [ ]:
# TRAINING VISUALIZATION
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.legend()
plt.title("Training vs Validation Accuracy")
plt.show()

In [ ]:
# CNN-LIKE APPROACH (USING 1D CONV ON TABULAR DATA)
from tensorflow.keras.layers import Conv1D, Flatten

# Reshape data for CNN (samples, timesteps, features)
X_train_cnn = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_cnn = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

cnn_model = Sequential()
cnn_model.add(Conv1D(filters=32, kernel_size=2, activation='relu', input_shape=(X_train.shape[1], 1)))
cnn_model.add(Conv1D(filters=16, kernel_size=2, activation='relu'))
cnn_model.add(Flatten())
cnn_model.add(Dense(32, activation='relu'))
cnn_model.add(Dense(2, activation='softmax'))

cnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])



In [ ]:
# FINAL EVALUATION
loss, acc = model.evaluate(X_test, y_test_cat)

print("\nNeural Network Accuracy:", acc)

In [ ]:
# FINAL SUMMARY
print("\n===== FINAL RESULTS =====")
for k, v in results.items():
    print(k, ":", v)

print("Best Random Forest:", accuracy_score(y_test, y_pred_best))
print("Neural Network:", acc)
